# Climate Downscaling - Studi Kasus: Yogyakarta
### Statistical Downscaling menggunakan Multiple Linear Regression (MLR)
**Studi Kasus:** Suhu / Hujan Harian Kota Yogyakarta  
**Metode:** Point-based downscaling ERA5 → Observasi BMKG → Proyeksi GCM

---
**Alur:**
```
1. Generate demo data (ERA5, Obs BMKG, GCM)
2. Eksplorasi data
3. Training MLR model
4. Validasi model
5. Proyeksi iklim masa depan
```

## 0. Import Library

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Style plot
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

np.random.seed(42)
print('✅ Library berhasil diimport')

---
## 1. Generate Demo Data

> **Catatan:** Dalam penelitian nyata, data ini diganti dengan:
> - ERA5 dari Copernicus CDS (`cdsapi`)
> - Observasi dari BMKG Stasiun Yogyakarta
> - GCM dari CMIP6 (misal MPI-ESM1-2-LR, skenario SSP2-4.5)

In [ ]:
# ============================================================
# PERIODE DATA
# ============================================================
# Historical: 2000-2020 (ERA5 + Obs BMKG)
# Future    : 2021-2060 (GCM proyeksi)

dates_hist   = pd.date_range('2000-01-01', '2020-12-31', freq='D')
dates_future = pd.date_range('2021-01-01', '2060-12-31', freq='D')
n_hist   = len(dates_hist)
n_future = len(dates_future)

print(f'Jumlah hari historis : {n_hist:,} hari ({dates_hist[0].date()} s/d {dates_hist[-1].date()})')
print(f'Jumlah hari future   : {n_future:,} hari ({dates_future[0].date()} s/d {dates_future[-1].date()})')

In [ ]:
# ============================================================
# FUNGSI HELPER: siklus musiman harian
# ============================================================
def seasonal_cycle(dates, amplitude=2.5, base=27.0):
    """Buat siklus musiman untuk suhu harian Yogyakarta."""
    doy = dates.day_of_year
    return base + amplitude * np.sin(2 * np.pi * (doy - 80) / 365)

# ============================================================
# 1A. ERA5 HISTORICAL (near-surface variables, titik nearest)
# ============================================================
t_seasonal_hist = seasonal_cycle(dates_hist)

era5_hist = pd.DataFrame({
    'date'   : dates_hist,
    't2m'    : t_seasonal_hist + np.random.normal(0, 1.2, n_hist),        # Suhu 2m (°C)
    'd2m'    : t_seasonal_hist - 4 + np.random.normal(0, 1.0, n_hist),    # Dewpoint (°C)
    'sp'     : 920 + np.random.normal(0, 3.5, n_hist),                    # Surface pressure (hPa)
    'ssrd'   : 200 + 60*np.sin(2*np.pi*(dates_hist.day_of_year-80)/365)
               + np.random.normal(0, 25, n_hist),                          # Solar radiation (W/m²)
    'u10'    : np.random.normal(1.5, 0.8, n_hist),                        # U-wind (m/s)
    'v10'    : np.random.normal(0.5, 0.6, n_hist),                        # V-wind (m/s)
})

# Tambah wind speed sebagai prediktor tambahan
era5_hist['ws10'] = np.sqrt(era5_hist['u10']**2 + era5_hist['v10']**2)

print('ERA5 Historical:')
print(era5_hist.describe().round(2))

In [ ]:
# ============================================================
# 1B. OBSERVASI BMKG (suhu harian stasiun Yogyakarta)
# ============================================================
# Obs = fungsi dari ERA5 + noise lokal
# Dalam kenyataan: download dari BMKG atau SIKU

obs_bmkg = pd.DataFrame({
    'date'     : dates_hist,
    'suhu_obs' : (0.85 * era5_hist['t2m']          # korelasi dengan T2M
                + 0.10 * era5_hist['ssrd'] / 60     # efek radiasi
                - 0.05 * era5_hist['ws10']           # efek angin mendinginkan
                + 2.3                                # bias lokal (urban heat island dll)
                + np.random.normal(0, 0.8, n_hist))  # noise residual lokal
})

print('Observasi BMKG:')
print(obs_bmkg['suhu_obs'].describe().round(2))

In [ ]:
# ============================================================
# 1C. GCM FUTURE (proyeksi SSP2-4.5, misal MPI-ESM)
# ============================================================
# GCM punya bias sendiri, ditambah tren pemanasan global

t_seasonal_fut = seasonal_cycle(dates_future)
# Tren pemanasan linear ~0.03°C/tahun
warming_trend  = np.linspace(0, 0.03 * 40, n_future)

gcm_future = pd.DataFrame({
    'date'   : dates_future,
    't2m'    : t_seasonal_fut + warming_trend + 1.5           # GCM warm bias +1.5°C
               + np.random.normal(0, 1.5, n_future),
    'd2m'    : t_seasonal_fut - 3.5 + warming_trend * 0.8
               + np.random.normal(0, 1.2, n_future),
    'sp'     : 922 + np.random.normal(0, 4.0, n_future),      # GCM pressure sedikit beda
    'ssrd'   : 205 + 60*np.sin(2*np.pi*(dates_future.day_of_year-80)/365)
               + np.random.normal(0, 28, n_future),
    'ws10'   : np.abs(np.random.normal(2.0, 0.9, n_future)),
})

print('GCM Future:')
print(gcm_future.describe().round(2))

---
## 2. Eksplorasi Data (EDA)

In [ ]:
# ============================================================
# PLOT 1: Time series ERA5 vs Observasi BMKG
# ============================================================
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=False)

# Full time series
axes[0].plot(era5_hist['date'], era5_hist['t2m'], 
             alpha=0.6, lw=0.8, color='steelblue', label='ERA5 T2M')
axes[0].plot(obs_bmkg['date'], obs_bmkg['suhu_obs'], 
             alpha=0.6, lw=0.8, color='tomato', label='Obs BMKG')
axes[0].set_title('Time Series Suhu Harian: ERA5 vs Observasi BMKG (2000–2020)', fontsize=13)
axes[0].set_ylabel('Suhu (°C)')
axes[0].legend()

# Zoom 1 tahun
mask = (era5_hist['date'] >= '2010-01-01') & (era5_hist['date'] <= '2010-12-31')
axes[1].plot(era5_hist[mask]['date'], era5_hist[mask]['t2m'], 
             lw=1.5, color='steelblue', label='ERA5 T2M')
axes[1].plot(obs_bmkg[mask]['date'], obs_bmkg[mask]['suhu_obs'], 
             lw=1.5, color='tomato', label='Obs BMKG')
axes[1].set_title('Zoom: Tahun 2010', fontsize=13)
axes[1].set_ylabel('Suhu (°C)')
axes[1].legend()

plt.tight_layout()
plt.savefig('plot_timeseries.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Plot time series selesai')

In [ ]:
# ============================================================
# PLOT 2: Korelasi prediktor vs target
# ============================================================
df_merged = pd.merge(era5_hist, obs_bmkg, on='date')

predictors = ['t2m', 'd2m', 'sp', 'ssrd', 'ws10']
pred_labels = ['T2M ERA5', 'Dewpoint', 'Sfc Pressure', 'Solar Rad', 'Wind Speed']

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for i, (pred, label) in enumerate(zip(predictors, pred_labels)):
    axes[i].scatter(df_merged[pred], df_merged['suhu_obs'], 
                    alpha=0.15, s=5, color='steelblue')
    # Trend line
    z = np.polyfit(df_merged[pred], df_merged['suhu_obs'], 1)
    p = np.poly1d(z)
    x_line = np.linspace(df_merged[pred].min(), df_merged[pred].max(), 100)
    axes[i].plot(x_line, p(x_line), 'r-', lw=2)
    r = df_merged[[pred, 'suhu_obs']].corr().iloc[0,1]
    axes[i].set_title(f'{label}\nr = {r:.3f}', fontsize=11)
    axes[i].set_xlabel(label)
    axes[i].set_ylabel('Obs BMKG (°C)')

axes[5].set_visible(False)
plt.suptitle('Korelasi Prediktor ERA5 vs Observasi BMKG', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('plot_korelasi.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# PLOT 3: Heatmap korelasi
# ============================================================
corr_cols = predictors + ['suhu_obs']
corr_labels = pred_labels + ['Obs BMKG']
corr_matrix = df_merged[corr_cols].corr()
corr_matrix.columns = corr_labels
corr_matrix.index   = corr_labels

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdYlBu_r',
            center=0, square=True, ax=ax, linewidths=0.5)
ax.set_title('Heatmap Korelasi Antar Variabel', fontsize=13)
plt.tight_layout()
plt.savefig('plot_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 3. Training Model MLR

**Persamaan:**
$$T_{obs} = \beta_0 + \beta_1 T_{2m} + \beta_2 T_{dew} + \beta_3 P_{sfc} + \beta_4 SSRD + \beta_5 WS + \epsilon$$

In [ ]:
# ============================================================
# PERSIAPAN DATA
# ============================================================
X = df_merged[predictors].values
y = df_merged['suhu_obs'].values

# Split train/test — shuffle=False karena time series!
split_idx = int(len(X) * 0.7)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]
dates_test = df_merged['date'].values[split_idx:]

print(f'Training : {X_train.shape[0]:,} sampel ({df_merged["date"].iloc[0].date()} s/d {df_merged["date"].iloc[split_idx-1].date()})')
print(f'Testing  : {X_test.shape[0]:,} sampel ({df_merged["date"].iloc[split_idx].date()} s/d {df_merged["date"].iloc[-1].date()})')

In [ ]:
# ============================================================
# STANDARDISASI (opsional tapi good practice)
# ============================================================
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# ============================================================
# FIT MODEL
# ============================================================
model = LinearRegression()
model.fit(X_train_sc, y_train)

print('✅ Model berhasil ditraining')
print('\n📊 Koefisien Regresi:')
print(f'  Intercept (β0) : {model.intercept_:.4f}°C')
for name, coef in zip(pred_labels, model.coef_):
    print(f'  {name:<15} : {coef:+.4f}')

---
## 4. Validasi Model

In [ ]:
# ============================================================
# METRIK VALIDASI
# ============================================================
y_pred_train = model.predict(X_train_sc)
y_pred_test  = model.predict(X_test_sc)

def print_metrics(y_true, y_pred, label):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    bias = np.mean(y_pred - y_true)
    print(f'  {label}')
    print(f'    RMSE  : {rmse:.3f} °C')
    print(f'    MAE   : {mae:.3f} °C')
    print(f'    R²    : {r2:.3f}')
    print(f'    Bias  : {bias:+.3f} °C')
    return rmse, mae, r2, bias

print('📈 Metrik Performa Model:')
m_train = print_metrics(y_train, y_pred_train, 'Training set')
print()
m_test  = print_metrics(y_test,  y_pred_test,  'Testing set')

In [ ]:
# ============================================================
# PLOT VALIDASI
# ============================================================
fig = plt.figure(figsize=(16, 10))
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.35, wspace=0.3)

# --- Plot 1: Time series test period ---
ax1 = fig.add_subplot(gs[0, :])
ax1.plot(dates_test, y_test,      lw=0.8, alpha=0.7, color='tomato',    label='Obs BMKG')
ax1.plot(dates_test, y_pred_test, lw=0.8, alpha=0.7, color='steelblue', label='MLR Pred')
ax1.set_title(f'Test Period: Obs vs MLR Prediction  (R²={m_test[2]:.3f}, RMSE={m_test[0]:.3f}°C)', fontsize=12)
ax1.set_ylabel('Suhu (°C)')
ax1.legend()

# --- Plot 2: Scatter obs vs pred ---
ax2 = fig.add_subplot(gs[1, 0])
ax2.scatter(y_test, y_pred_test, alpha=0.2, s=5, color='steelblue')
lims = [min(y_test.min(), y_pred_test.min())-0.5, 
        max(y_test.max(), y_pred_test.max())+0.5]
ax2.plot(lims, lims, 'r--', lw=1.5, label='1:1 line')
ax2.set_xlabel('Obs BMKG (°C)')
ax2.set_ylabel('MLR Prediction (°C)')
ax2.set_title('Scatter: Obs vs Pred', fontsize=12)
ax2.legend()

# --- Plot 3: Residual distribution ---
ax3 = fig.add_subplot(gs[1, 1])
residuals = y_test - y_pred_test
ax3.hist(residuals, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
ax3.axvline(0, color='red', lw=1.5, linestyle='--')
ax3.set_xlabel('Residual (°C)')
ax3.set_ylabel('Frekuensi')
ax3.set_title(f'Distribusi Residual (μ={residuals.mean():.3f}, σ={residuals.std():.3f})', fontsize=12)

plt.suptitle('Validasi Model MLR — Test Period', fontsize=14, y=1.01)
plt.savefig('plot_validasi.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# PERBANDINGAN DISTRIBUSI (sebelum & sesudah koreksi)
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Histogram
axes[0].hist(y_test,       bins=50, alpha=0.6, color='tomato',    label='Obs BMKG',   density=True)
axes[0].hist(X_test[:,0] * scaler.scale_[0] + scaler.mean_[0],  
             bins=50, alpha=0.6, color='gray',     label='ERA5 raw',   density=True)
axes[0].hist(y_pred_test,  bins=50, alpha=0.6, color='steelblue', label='MLR corrected', density=True)
axes[0].set_title('Distribusi Suhu: Before vs After Downscaling', fontsize=11)
axes[0].set_xlabel('Suhu (°C)')
axes[0].set_ylabel('Density')
axes[0].legend()

# Boxplot
box_data = [
    y_test,
    X_test[:,0] * scaler.scale_[0] + scaler.mean_[0],
    y_pred_test
]
bp = axes[1].boxplot(box_data, patch_artist=True,
                      labels=['Obs BMKG', 'ERA5 raw', 'MLR Pred'])
colors = ['tomato', 'gray', 'steelblue']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[1].set_title('Boxplot Perbandingan', fontsize=11)
axes[1].set_ylabel('Suhu (°C)')

plt.tight_layout()
plt.savefig('plot_distribusi.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 5. Proyeksi GCM Future

> Di sini kita aplikasikan model MLR yang sudah ditraining ke prediktor GCM masa depan.

In [ ]:
# ============================================================
# SIAPKAN PREDIKTOR GCM FUTURE
# ============================================================
# Pastikan urutan kolom sama dengan training!
X_future = gcm_future[predictors].values
X_future_sc = scaler.transform(X_future)  # pakai scaler yang sama!

# Proyeksi
gcm_future['suhu_downscaled'] = model.predict(X_future_sc)

print('✅ Proyeksi selesai')
print(f'Rentang suhu proyeksi: {gcm_future["suhu_downscaled"].min():.1f}°C – {gcm_future["suhu_downscaled"].max():.1f}°C')

In [ ]:
# ============================================================
# HITUNG ANOMALI (perubahan dari baseline)
# ============================================================
# Baseline: rata-rata historis obs BMKG (2000-2020)
baseline_mean = obs_bmkg['suhu_obs'].mean()

# Rata-rata tahunan proyeksi
gcm_future['year'] = gcm_future['date'].dt.year
annual_proj = gcm_future.groupby('year')['suhu_downscaled'].mean().reset_index()
annual_proj['anomali'] = annual_proj['suhu_downscaled'] - baseline_mean

# Juga hitung historis untuk plot kontinu
obs_bmkg['year'] = obs_bmkg['date'].dt.year
annual_hist = obs_bmkg.groupby('year')['suhu_obs'].mean().reset_index()

print(f'Baseline mean (2000–2020): {baseline_mean:.2f}°C')
print(f'Proyeksi akhir (2056–2060): {annual_proj[annual_proj["year"]>=2056]["suhu_downscaled"].mean():.2f}°C')
print(f'Anomali di akhir periode  : +{annual_proj[annual_proj["year"]>=2056]["anomali"].mean():.2f}°C')

In [ ]:
# ============================================================
# PLOT PROYEKSI UTAMA
# ============================================================
fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=False)

# --- Plot atas: suhu absolut ---
ax = axes[0]
ax.plot(annual_hist['year'], annual_hist['suhu_obs'],
        color='tomato', lw=2, marker='o', ms=3, label='Obs BMKG (historis)')
ax.plot(annual_proj['year'], annual_proj['suhu_downscaled'],
        color='steelblue', lw=2, label='Proyeksi (MLR downscaled)')

# Smoothing proyeksi (rolling 5 tahun)
proj_smooth = annual_proj['suhu_downscaled'].rolling(5, center=True).mean()
ax.plot(annual_proj['year'], proj_smooth, 
        color='navy', lw=2.5, linestyle='--', label='Proyeksi (5-yr smooth)')

ax.axhline(baseline_mean, color='gray', lw=1, linestyle=':', alpha=0.8, label=f'Baseline ({baseline_mean:.1f}°C)')
ax.axvline(2020.5, color='black', lw=1.5, linestyle='--', alpha=0.5)
ax.text(2021, ax.get_ylim()[0]+0.1, '← Historis | Future →', fontsize=9, color='gray')
ax.set_ylabel('Suhu Rata-rata Tahunan (°C)', fontsize=11)
ax.set_title('Proyeksi Suhu Harian Yogyakarta — MLR Downscaling GCM (SSP2-4.5)', fontsize=13)
ax.legend(loc='upper left')

# --- Plot bawah: anomali ---
ax2 = axes[1]
colors_bar = ['tomato' if v > 0 else 'steelblue' for v in annual_proj['anomali']]
ax2.bar(annual_proj['year'], annual_proj['anomali'], color=colors_bar, alpha=0.7)
ax2.axhline(0, color='black', lw=1)

# Trend line anomali
z = np.polyfit(annual_proj['year'], annual_proj['anomali'], 1)
p = np.poly1d(z)
ax2.plot(annual_proj['year'], p(annual_proj['year']), 'k--', lw=2, 
         label=f'Tren: {z[0]*10:+.3f}°C/dekade')
ax2.set_ylabel('Anomali Suhu (°C)', fontsize=11)
ax2.set_xlabel('Tahun', fontsize=11)
ax2.set_title('Anomali terhadap Baseline 2000–2020', fontsize=12)
ax2.legend()

plt.tight_layout()
plt.savefig('plot_proyeksi.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# PLOT: Siklus musiman historis vs future
# ============================================================
obs_bmkg['month']        = obs_bmkg['date'].dt.month
gcm_future['month']      = gcm_future['date'].dt.month

# Dua periode future
near_future = gcm_future[gcm_future['year'].between(2021, 2040)]
far_future  = gcm_future[gcm_future['year'].between(2041, 2060)]

monthly_hist  = obs_bmkg.groupby('month')['suhu_obs'].mean()
monthly_near  = near_future.groupby('month')['suhu_downscaled'].mean()
monthly_far   = far_future.groupby('month')['suhu_downscaled'].mean()

months = ['Jan','Feb','Mar','Apr','Mei','Jun','Jul','Agu','Sep','Okt','Nov','Des']

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(range(1,13), monthly_hist.values,  'o-', lw=2, color='tomato',    label='Historis (2000–2020)')
ax.plot(range(1,13), monthly_near.values,  's-', lw=2, color='steelblue', label='Near Future (2021–2040)')
ax.plot(range(1,13), monthly_far.values,   '^-', lw=2, color='navy',      label='Far Future (2041–2060)')
ax.fill_between(range(1,13), monthly_hist.values, monthly_far.values, alpha=0.1, color='navy')
ax.set_xticks(range(1,13))
ax.set_xticklabels(months)
ax.set_ylabel('Suhu (°C)')
ax.set_title('Siklus Musiman Suhu: Historis vs Proyeksi (Yogyakarta)', fontsize=13)
ax.legend()
plt.tight_layout()
plt.savefig('plot_musiman.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 6. Ringkasan Hasil

In [ ]:
# ============================================================
# RINGKASAN
# ============================================================
print('=' * 55)
print('  RINGKASAN: DOWNSCALING SUHU YOGYAKARTA')
print('=' * 55)
print()
print('📌 Model       : Multiple Linear Regression (MLR)')
print(f'📌 Prediktor   : {pred_labels}')
print(f'📌 Training    : 2000–2014 ({X_train.shape[0]:,} hari)')
print(f'📌 Testing     : 2015–2020 ({X_test.shape[0]:,} hari)')
print()
print('📊 Performa Model (Test Period):')
print(f'   R²   = {m_test[2]:.3f}')
print(f'   RMSE = {m_test[0]:.3f} °C')
print(f'   MAE  = {m_test[1]:.3f} °C')
print(f'   Bias = {m_test[3]:+.3f} °C')
print()
print('🌡️  Proyeksi Suhu Yogyakarta (SSP2-4.5):')
print(f'   Baseline (2000–2020)  : {baseline_mean:.2f} °C')
print(f'   Near future (2021–2040): {near_future["suhu_downscaled"].mean():.2f} °C')
print(f'   Far future  (2041–2060): {far_future["suhu_downscaled"].mean():.2f} °C')
trend_per_decade = z[0] * 10
print(f'   Tren pemanasan         : {trend_per_decade:+.3f} °C/dekade')
print()
print('⚠️  Catatan untuk Skripsi:')
print('   - Ganti demo data dengan ERA5 real (cdsapi)')
print('   - Data obs dari BMKG Yogyakarta (minta langsung)')
print('   - GCM dari CMIP6 (esgf-node.llnl.gov)')
print('   - Bisa dicoba juga Ridge/Lasso regression')
print('=' * 55)

---
## 7. Export Hasil ke CSV

In [ ]:
# Export proyeksi harian
gcm_future[['date', 'suhu_downscaled']].to_csv('proyeksi_suhu_yogyakarta.csv', index=False)

# Export proyeksi tahunan
annual_proj.to_csv('proyeksi_tahunan_yogyakarta.csv', index=False)

print('✅ Data proyeksi tersimpan:')
print('   - proyeksi_suhu_yogyakarta.csv  (harian)')
print('   - proyeksi_tahunan_yogyakarta.csv (tahunan)')